In [ ]:
using Pkg
toktox_path = "/Users/adrianaghiozzi/Code/Sources/dev/TokToxStability"   # adjust to match the location of your cloned TokToxStability repo
Pkg.activate(toktox_path)
Pkg.instantiate()

In [ ]:
venv_python = joinpath(toktox_path, ".venv", "bin", "python")
if !isfile(venv_python)
    run(`$(joinpath(toktox_path, "setup_venv.sh"))`)
end

In [ ]:
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = venv_python

In [ ]:
using Revise
using TokToxStability
using Plots
using DataFrames
using PythonCall

In [ ]:
myOFT = TokToxStability.make_oft_env()   # only call this ONCE per kernel session


In [ ]:
gpec_settings = TokToxStability.GPECSettings(toroidal_mode_number = 3)
kinetic_efit_path = joinpath(pkgdir(TokToxStability), "cases/D3D/g147131.02300_DIIID_KEFIT_corrected.kin")


In [ ]:
profiles = TokToxStability.read_kinetic_profiles_from_file(kinetic_efit_path)
equilibrium = TokToxStability.read_equilibrium_from_file(kinetic_efit_path)


In [ ]:
mygs = TokToxStability.setup_tokamaker_from_file!(:D3D, profiles, equilibrium, myOFT)


In [ ]:
coarse_scan_results = TokToxStability.coarse_scan_until_unstable(mygs, 1.0, 0.25, equilibrium, profiles, gpec_settings; working_directory = ".");


In [ ]:
cols = (:fraction, :beta_n, :dW_total, :dW_plasma, :dW_vacuum, :qmax, :msing)
DataFrame([NamedTuple{cols}(r[c] for c in cols) for r in coarse_scan_results])

In [ ]:
bl = TokToxStability.estimate_beta_limit(coarse_scan_results)
TokToxStability.plot_beta_scan_summary(coarse_scan_results; beta_limit=bl)


In [ ]:
TokToxStability.plot_profiles(coarse_scan_results)


In [ ]:
TokToxStability.plot_q_profile(coarse_scan_results)
